## ▶ Run Online — No Installation Needed

| Platform | Link |
|---|---|
| **Binder** (no account) | [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Piyushjhu/HELIX_Toolbox/main?labpath=examples%2F01_full_pipeline_cli.ipynb) |
| **Google Colab** | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Piyushjhu/HELIX_Toolbox/blob/main/examples/01_full_pipeline_cli.ipynb) |
| **GitHub Codespaces** | [![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/Piyushjhu/HELIX_Toolbox) |

**Using your own data?** Run the setup cell below — it will show an upload widget when running in Binder or Colab. On Codespaces, drag-and-drop your CSVs into the file explorer then update the path variables in the next cell.

In [ ]:

# ── Cloud / Online environment setup ──────────────────────────────────────
# This cell auto-detects Binder, Google Colab, GitHub Codespaces, and local
# environments.  It installs dependencies, sets headless Qt/matplotlib, and
# wires REPO_ROOT so all imports work without changes to subsequent cells.
# ──────────────────────────────────────────────────────────────────────────
import os, sys, subprocess

# ── Detect environment ─────────────────────────────────────────────────────
try:
    import google.colab
    _ENV = "colab"
except ImportError:
    _ENV = "binder" if os.environ.get("BINDER_SERVICE_HOST") else "local"

print(f"Detected environment: {_ENV}")

# ── Install / configure ────────────────────────────────────────────────────
if _ENV in ("colab", "binder"):
    # Headless Qt (no display needed) and non-interactive matplotlib
    os.environ["QT_QPA_PLATFORM"] = "offscreen"
    os.environ["MPLBACKEND"] = "Agg"

if _ENV == "colab":
    # Fixed absolute path — prevents double/triple nesting when cell is re-run
    REPO_ROOT = "/content/HELIX_Toolbox"
    if not os.path.isdir(REPO_ROOT):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Piyushjhu/HELIX_Toolbox.git",
             REPO_ROOT],
            check=True
        )
    # Install dependencies using the absolute path — no os.chdir() needed
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         os.path.join(REPO_ROOT, "requirements.txt")],
        check=False
    )
elif _ENV == "binder":
    # On Binder the repo is already at the working directory root
    REPO_ROOT = os.path.abspath("..")
else:
    # Local — adjust if your notebook is not inside the examples/ subfolder
    REPO_ROOT = os.path.abspath("..")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# ── Bundled sample data (used when no local files are provided) ────────────
SAMPLE_DIR = os.path.join(REPO_ROOT, "input_data", "C1_files")
_sample_files = sorted(f for f in os.listdir(SAMPLE_DIR) if f.endswith(".csv")) if os.path.isdir(SAMPLE_DIR) else []
print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"Sample data: {SAMPLE_DIR}")
print(f"  files    : {_sample_files}")

# ── Upload widget (Binder / Colab only) ────────────────────────────────────
_uploaded_dir = None
if _ENV == "colab":
    print("\nTo upload YOUR OWN data:  from google.colab import files; uploaded = files.upload()")
elif _ENV == "binder":
    print("\nTo use YOUR OWN data, use the Jupyter file-browser (left panel) to")
    print("upload CSVs, then set INPUT_DIR in the next cell to point to them.")
else:
    print("\nRunning locally — edit path variables in the next cell as needed.")


# Example 1 — Full Pipeline: Raw PDV → Spall Strength (CLI)

This notebook walks through a complete end-to-end HELIX Toolbox run:

```
Raw PDV oscilloscope CSV
        ↓  ALPSS
Smoothed velocity trace + uncertainty CSV
        ↓  SPADE
Spall strength / strain rate / shock stress / HEL summary CSV + plots
```

The run is driven entirely by a YAML config file — the same workflow you
would use in batch / HPC mode via the CLI:
```bash
python helix_cli_runner.py --config my_experiment.yml
```

---
**Before running:** update the path variables in the next cell.

In [ ]:
import os, sys

# ── USER PATHS — edit these for local runs ────────────────────────────────
# On Binder/Colab, REPO_ROOT and SAMPLE_DIR are already set by the setup cell
# above.  Override INPUT_DIR here to point to your own uploaded files.
try:
    REPO_ROOT   # set by setup cell
except NameError:
    REPO_ROOT = os.path.abspath("..")

try:
    SAMPLE_DIR  # set by setup cell
except NameError:
    SAMPLE_DIR = os.path.join(REPO_ROOT, "input_data", "C1_files")

# Default to bundled sample data; replace with your own folder path:
INPUT_DIR    = SAMPLE_DIR                    # ← your PDV CSV folder
PARAM_FOLDER = None                          # ← metadata xlsx/csv folder (or None)
OUTPUT_DIR   = os.path.join(REPO_ROOT, "examples", "figures", "example_01_output")
# ──────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("Repo root :", REPO_ROOT)
print("Input dir :", INPUT_DIR)
print("Output dir:", OUTPUT_DIR)

## 1. Build the config programmatically

You can write a config dict in Python and save it as a YAML file,
or just point to your existing `helix_master_config.yml`.

In [ ]:
from helix_analysis_toolbox import save_config_to_file, load_config_from_file

config = {
    "cli_settings": {
        "input_dir":      INPUT_DIR,
        "input_pattern":  "*.csv",
        "output_dir":     OUTPUT_DIR,
        "param_folder":   PARAM_FOLDER,
        "analysis_mode":  "both",       # both | alpss_only | spade_only
        "spade_mode":     "auto",
        "input_files":    None,
        "spade_input_files": None,
        "spade_input_dir":   None,
        "spade_input_pattern": "*--vel-smooth-with-uncert.csv",
    },
    "alpss_config": {
        "save_data":               "yes",
        "display_plots":           "no",
        "save_all_plots":          "no",
        "header_lines":            22,
        "start_time_user":         "none",
        "start_time_correction":   0,
        "time_to_skip":            0e-6,
        "time_to_take":            4e-6,
        "t_before":                2e-8,
        "t_after":                 2e-7,
        "use_robust_iq_detection": True,
        "iq_threshold_factor":     0.8,
        "iq_smoothing_window_ns":  5.0,
        "iq_skip_start_ns":        1.0,
        "iq_persistence_ns":       0.5,
        "freq_min":                1.2e9,
        "freq_max":                3.0e9,
        "smoothing_type":          "savgol",
        "smoothing_window":        None,
        "smoothing_window_ns":     6.0,
        "smoothing_wid":           3.0,
        "smoothing_amp":           1.0,
        "smoothing_sigma":         1.0,
        "smoothing_mu":            0.0,
        "savgol_polyorder":        3,
        "use_notch_filter":        False,
        "sample_rate":             1.28e11,
        "nperseg":                 512,
        "noverlap":                400,
        "nfft":                    2560,
        "window":                  "hann",
        "blur_kernel_x":           3,
        "blur_kernel_y":           3,
        "blur_sigx":               0.0,
        "blur_sigy":               0.0,
        "save_velocity_smooth_uncert_csv": True,
        "save_results_csv":        True,
        "C0":      5020.0,    # CP-Ti bulk wave speed (m/s)
        "density": 4510.0,    # CP-Ti density (kg/m³)
        "lam":     1.55e-6,
        "theta":   0.0,
    },
    "spade_config": {
        "experiment_velocity_shots": True,
        "experiment_spall_analysis": True,
        "experiment_hel_detection":  True,
        "density":           4510.0,   # CP-Ti
        "acoustic_velocity": 5020.0,   # CP-Ti bulk wave speed (m/s)
        "analysis_model":            "hybrid",
        "spall_detection_method":    "5-segment",
        "spall_start_time_ns":       20.0,
        "spall_end_time_ns":         60.0,
        "threshold_velocity_ms":     5.0,
        "hel_start_time_ns":         0,
        "hel_end_time_ns":           20,
        "minimum_HEL_velocity_expected": 40.0,
        "hel_rdp_epsilon":           1.0,
        "mad_filter_enabled":        True,
        "mad_filter_threshold":      2.0,
        "skip_unknown_material_traces": False,
        "plot_individual":           True,
        "save_summary":              True,
        "show_plots":                False,
    },
    "material_properties": {
        "Ti":       {"density": 4510.0, "bulk_wave_speed": 5020.0, "C0": 5020.0, "C_L": 6070.0},
        "CP-Ti":    {"density": 4510.0, "bulk_wave_speed": 5020.0, "C0": 5020.0, "C_L": 6070.0},
        "Ti64":     {"density": 4430.0, "bulk_wave_speed": 5130.0, "C0": 5130.0, "C_L": 6130.0},
        "Ti-6Al-4V":{"density": 4430.0, "bulk_wave_speed": 5130.0, "C0": 5130.0, "C_L": 6130.0},
        "Cu":       {"density": 8960.0, "bulk_wave_speed": 3950.0, "C0": 3950.0, "C_L": 4700.0},
        "Zn":       {"density": 7140.0, "bulk_wave_speed": 3700.0, "C0": 3700.0, "C_L": 4200.0},
    },
}

config_path = os.path.join(OUTPUT_DIR, "run_config.yml")
ok, msg = save_config_to_file(config, config_path)
print(msg)

## 2. Run the CLI

We call `helix_cli_runner.py` as a subprocess so the output streams live
to the notebook cell output exactly as it would in a terminal.

In [ ]:
import subprocess, os

_runner = os.path.join(REPO_ROOT, "helix_cli_runner.py")
print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"runner path : {_runner}  (exists={os.path.isfile(_runner)})")
print(f"config path : {config_path}  (exists={os.path.isfile(config_path)})")
print("-" * 60)

result = subprocess.run(
    [sys.executable, _runner, "--config", config_path],
    capture_output=True,    # capture so we can print stderr on failure
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("── STDERR ──────────────────────────────────────────────────")
    print(result.stderr)
print(f"\nExit code: {result.returncode}")

## 3. Inspect the output summary

In [ ]:
import pandas as pd

spade_dir = os.path.join(OUTPUT_DIR, "SPADE_analysis")
summary_path = os.path.join(spade_dir, "velocity_shots_summary.csv")

if os.path.exists(summary_path):
    df = pd.read_csv(summary_path)
    print(f"Loaded {len(df)} traces from {summary_path}")
    display(df.head(10))
else:
    print(f"Summary not found at {summary_path} — check run output above for errors.")

## 4. Quick result plots

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if 'df' not in dir() or df is None:
    print("Run the cell above first to load the summary CSV.")
else:
    # Try common column name variants
    def _find(df, *candidates):
        for c in candidates:
            for col in df.columns:
                if c.lower() in col.lower():
                    return col
        return None

    col_stress  = _find(df, 'shock_stress', 'Shock_Stress')
    col_spall   = _find(df, 'spall_strength', 'Spall_Strength')
    col_strrate = _find(df, 'strain_rate', 'Strain_Rate')
    col_mat     = _find(df, 'material', 'Material', 'Sample')

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Spall strength vs strain rate
    if col_spall and col_strrate:
        ax = axes[0]
        if col_mat:
            for mat, grp in df.groupby(col_mat):
                ax.scatter(grp[col_strrate], grp[col_spall], label=mat, s=60)
            ax.legend()
        else:
            ax.scatter(df[col_strrate], df[col_spall], s=60)
        ax.set_xlabel("Strain rate (s⁻¹)")
        ax.set_ylabel("Spall strength (GPa)")
        ax.set_title("Spall strength vs strain rate")
        ax.set_xscale("log")

    # Shock stress vs spall strength
    if col_stress and col_spall:
        ax = axes[1]
        if col_mat:
            for mat, grp in df.groupby(col_mat):
                ax.scatter(grp[col_stress], grp[col_spall], label=mat, s=60)
            ax.legend()
        else:
            ax.scatter(df[col_stress], df[col_spall], s=60)
        ax.set_xlabel("Shock stress (GPa)")
        ax.set_ylabel("Spall strength (GPa)")
        ax.set_title("Spall strength vs shock stress")

    plt.tight_layout()
    fig_path = os.path.join("figures", "example_01_summary_plots.png")
    os.makedirs("figures", exist_ok=True)
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {fig_path}")

## 5. Show a generated individual trace plot

HELIX saves a per-trace spall detection plot for every file when
`plot_individual: true`.  Pick one to display here.

In [ ]:
import glob as _glob
from IPython.display import Image, display as ipy_display

# Find the first spall plot
spall_plots = _glob.glob(os.path.join(spade_dir, "spall_plots", "*.png"))
if spall_plots:
    print(f"Found {len(spall_plots)} spall plots — showing the first one:")
    ipy_display(Image(spall_plots[0], width=800))
else:
    print("No spall plots found — check that plot_individual: true in the config.")